## 1. Import the packages

In [16]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

### Load and merge the training data

In [17]:
DATA_DIR = Path("../data_sets")

train_features_df = pd.read_csv(
    DATA_DIR / "training_set_features.csv"
)

train_labels_df = pd.read_csv(
    DATA_DIR / "training_set_labels.csv"
)

model_data_df = train_features_df.merge(
    train_labels_df,
    on="respondent_id",
    how="inner",
    validate="one_to_one"
)

print("Modeling dataset shape:", model_data_df.shape)

Modeling dataset shape: (26707, 38)


## 2: Feature engineering

### Preventive-behaviour score
These columns represent preventive actions:

In [18]:
preventive_behavior_columns = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face"
]

model_data_df["preventive_behavior_score"] = (
    model_data_df[preventive_behavior_columns]
    .sum(axis=1, min_count=1)
)

In [19]:
model_data_df["household_size"] = (
    model_data_df["household_adults"]
    + model_data_df["household_children"]
    + 1
)

### H1N1 risk-perception score

In [20]:
normalized_h1n1_concern = (
    model_data_df["h1n1_concern"] / 3
)

normalized_h1n1_risk = (
    model_data_df["opinion_h1n1_risk"] - 1
) / 4

model_data_df["h1n1_risk_perception_score"] = (
    pd.concat(
        [
            normalized_h1n1_concern,
            normalized_h1n1_risk
        ],
        axis=1
    )
    .mean(axis=1)
)

### H1N1 vaccine-confidence score

In [21]:
normalized_h1n1_effectiveness = (
    model_data_df["opinion_h1n1_vacc_effective"] - 1
) / 4

reversed_h1n1_sickness_fear = (
    5 - model_data_df["opinion_h1n1_sick_from_vacc"]
) / 4

model_data_df["h1n1_vaccine_confidence_score"] = (
    pd.concat(
        [
            normalized_h1n1_effectiveness,
            reversed_h1n1_sickness_fear
        ],
        axis=1
    )
    .mean(axis=1)
)

### Seasonal vaccine-confidence score

In [22]:
normalized_seasonal_effectiveness = (
    model_data_df["opinion_seas_vacc_effective"] - 1
) / 4

reversed_seasonal_sickness_fear = (
    5 - model_data_df["opinion_seas_sick_from_vacc"]
) / 4

model_data_df["seasonal_vaccine_confidence_score"] = (
    pd.concat(
        [
            normalized_seasonal_effectiveness,
            reversed_seasonal_sickness_fear
        ],
        axis=1
    )
    .mean(axis=1)
)

### Household-size feature

In [23]:
model_data_df["household_size"] = (
    model_data_df["household_adults"]
    + model_data_df["household_children"]
    + 1
)

### Medical-risk indicator

In [24]:
model_data_df["medical_risk_indicator"] = (
    model_data_df[
        [
            "chronic_med_condition",
            "child_under_6_months"
        ]
    ]
    .max(axis=1)
)

### Seasonal confidence

In [25]:
normalized_seasonal_effectiveness = (
    model_data_df["opinion_seas_vacc_effective"] - 1
) / 4

reversed_seasonal_sickness_fear = (
    5 - model_data_df["opinion_seas_sick_from_vacc"]
) / 4

model_data_df["seasonal_vaccine_confidence_score"] = (
    pd.concat(
        [
            normalized_seasonal_effectiveness,
            reversed_seasonal_sickness_fear
        ],
        axis=1
    )
    .mean(axis=1)
)

In [26]:
model_data_df["medical_risk_indicator"] = (
    model_data_df[
        [
            "chronic_med_condition",
            "child_under_6_months"
        ]
    ]
    .max(axis=1)
)

In [27]:
### Verification of the Engineered Features

In [28]:
engineered_features = [
    "preventive_behavior_score",
    "h1n1_risk_perception_score",
    "h1n1_vaccine_confidence_score",
    "seasonal_vaccine_confidence_score",
    "household_size",
    "medical_risk_indicator"
]


# Check whether every feature exists
missing_features = [
    feature
    for feature in engineered_features
    if feature not in model_data_df.columns
]

if missing_features:
    print("Missing engineered features:")
    print(missing_features)
else:
    print("All engineered features exist.")


# Display ranges, averages and missing values
feature_summary = (
    model_data_df[engineered_features]
    .describe()
    .T
)

feature_summary["missing_count"] = (
    model_data_df[engineered_features]
    .isna()
    .sum()
)

display(feature_summary)

All engineered features exist.


,count,mean,std,min,25%,50%,75%,max,missing_count
preventive_behavior_score,26707.0,3.029693,1.605258,0.0,2.000000,3.000000,4.000,7.0,0
h1n1_risk_perception_score,26700.0,0.438778,0.260973,0.0,0.291667,0.458333,0.625,1.0,7
h1n1_vaccine_confidence_score,26413.0,0.686641,0.206238,0.0,0.500000,0.750000,0.875,1.0,294
seasonal_vaccine_confidence_score,26290.0,0.738332,0.217487,0.0,0.500000,0.750000,0.875,1.0,417
household_size,26458.0,2.421082,1.301647,1.0,1.000000,2.000000,3.000,7.0,249
medical_risk_indicator,25982.0,0.339966,0.473706,0.0,0.000000,0.000000,1.000,1.0,725


### Verify 

In [29]:
range_checks = {
    "preventive_behavior_score": (0, 7),
    "h1n1_risk_perception_score": (0, 1),
    "h1n1_vaccine_confidence_score": (0, 1),
    "seasonal_vaccine_confidence_score": (0, 1),
    "household_size": (1, 7),
    "medical_risk_indicator": (0, 1)
}

for feature, (minimum, maximum) in range_checks.items():

    valid = (
        model_data_df[feature]
        .dropna()
        .between(minimum, maximum)
        .all()
    )

    print(
        feature,
        "is valid:" if valid else "has invalid values:",
        valid
    )

preventive_behavior_score is valid: True
h1n1_risk_perception_score is valid: True
h1n1_vaccine_confidence_score is valid: True
seasonal_vaccine_confidence_score is valid: True
household_size is valid: True
medical_risk_indicator is valid: True


## Separate features and targets

In [30]:
target_columns = [
    "h1n1_vaccine",
    "seasonal_vaccine"
]

X = model_data_df.drop(
    columns=[
        "respondent_id",
        "h1n1_vaccine",
        "seasonal_vaccine"
    ]
)

y_h1n1 = model_data_df["h1n1_vaccine"]
y_seasonal = model_data_df["seasonal_vaccine"]

### Verifying the separation

In [31]:
print("Predictor shape:", X.shape)
print("H1N1 target shape:", y_h1n1.shape)
print("Seasonal target shape:", y_seasonal.shape)

print(
    "\nTargets accidentally included in X:",
    set(target_columns).intersection(X.columns)
)

print(
    "Respondent ID included in X:",
    "respondent_id" in X.columns
)

Predictor shape: (26707, 41)
H1N1 target shape: (26707,)
Seasonal target shape: (26707,)

Targets accidentally included in X: set()
Respondent ID included in X: False


### Check the target distributions

In [32]:
print("H1N1 target distribution:")
display(
    y_h1n1
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Seasonal target distribution:")
display(
    y_seasonal
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

H1N1 target distribution:


h1n1_vaccine
0    78.75
1    21.25
Name: proportion, dtype: float64

Seasonal target distribution:


seasonal_vaccine
0    53.44
1    46.56
Name: proportion, dtype: float64